## 1. Instalación de dependencias

### Opcion 1: Si tenes una sola instalacion de Python

In [ ]:
# pip install curl_cffi beautifulsoup4 pandas

### Opcion 2: Si tenes varias instalaciones de Python

In [1]:
# # Ejecutar esta celda una sola vez para instalar los paquetes
# # Despues de instalarse, se reinicia el kernel para que esten disponibles en las siguientes celdas

# import sys, subprocess, importlib

# PACKAGES = ["curl_cffi", "beautifulsoup4", "pandas"]

# def install_and_restart(packages):
#     installed_any = False
#     for pkg in packages:
#         mod = pkg.split("[")[0].replace("-", "_")  
#         try:
#             importlib.import_module(mod)
#         except ImportError:
#             print(f"Installing {pkg}...")
#             subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
#             installed_any = True
    
#     if installed_any:
#         print("\n✅ Packages installed. Restarting kernel...")
#         import IPython
#         IPython.Application.instance().kernel.do_shutdown(restart=True)
#     else:
#         print("✅ All packages already installed, no restart needed.")

# install_and_restart(PACKAGES)

## 2. Imports y funciones auxiliares

In [ ]:
import re
import os
import time
import pandas as pd
from datetime import date
from bs4 import BeautifulSoup
from curl_cffi import requests as cf_requests

# ──────────────────────────────────────────────
# UTILIDADES
# ──────────────────────────────────────────────

def clean_text(text):
    if not text: return None
    text = text.replace('\xa0', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def parse_price(price_text):
    price_text = clean_text(price_text)
    precio = "Consultar"
    expensas = None
    p_match = re.search(r'(USD|ARS|\$)\s?[\$]?\s?([\d\.]+)', price_text)
    if p_match:
        precio = p_match.group(0).strip()
    e_match = re.search(r'[Ee]xp[^$\d]*([\d\.]+)', price_text)
    if not e_match:
        e_match = re.search(r'\+\s?\$?\s?([\d\.]+)', price_text)
    if e_match:
        expensas = e_match.group(0).strip()
    return precio, expensas

def extract_smart_features(row):
    texto = (str(row.get('Descripción', '')) + " " + str(row.get('Detalles', ''))).lower()
    return pd.Series({
        "Amenities":         sum(1 for x in ["amenities", "piscina", "pileta", "sum", "parrilla", "gym", "sauna", "laundry"] if x in texto),
        "Losa_Central":      1 if any(x in texto for x in ["losa radiante", "calefacción central", "caldera central", "piso radiante"]) else 0,
        "Aire_Acond":        1 if any(x in texto for x in ["aire acondicionado", "split", " a/c", "frío-calor"]) else 0,
        "Apto_Credito":      1 if "apto crédito" in texto or "apto credito" in texto else 0,
        "Cochera":           1 if any(x in texto for x in ["cochera", "espacio guarda coche", "estacionamiento", "guarda coche"]) else 0,
        "Seguridad":         1 if any(x in texto for x in ["vigilancia", "seguridad 24", "tótem", "totem", "encargado"]) else 0,
        "Luminoso":          1 if any(x in texto for x in ["luminoso", "todo luz", "vista abierta", "vista panorámica", "sol"]) else 0,
        "Balcon_Aterrazado": 1 if "aterrazado" in texto or "balcón terraza" in texto else 0,
    })


MESES = r'(enero|febrero|marzo|abril|mayo|junio|julio|agosto|septiembre|octubre|noviembre|diciembre)'

def parse_address(address_raw):
    """
    Maneja todos los casos conocidos de ZonaProp:
    - 'Bolivia al 4400'                          → calle=Bolivia, altura=4400
    - 'Av Luis María Campos 1500'                → calle=Av Luis María Campos, altura=1500
    - 'Junín 1615 piso 13'                       → calle=Junín, altura=1615, piso=13
    - 'SAN JOSE 445. Entre Belgrano y Venezuela' → calle=SAN JOSE, altura=445
    - '11 de Septiembre de 1888 2231'            → calle=11 de Septiembre de 1888, altura=2231
    - 'Torres del Yacht - Juana Manso al 600'    → calle=Juana Manso, altura=600
    - 'Alvear Tower - Azucena Villaflor'         → calle=None, altura=None (sin número)
    - 'El Faro - 3 Ambientes + Dependencia'      → calle=None, altura=None (número no es altura)
    """
    calle = altura = piso = None
    if not address_raw:
        return calle, altura, piso

    try:
        # 1. Limpiar "al " antes de números
        cleaned = re.sub(r'\bal\s+', '', address_raw, flags=re.IGNORECASE)

        # 2. Si tiene guión, quedarse con el fragmento que tenga número de altura válido
        #    Ej: "Alvear Tower - Azucena Villaflor" → ningún fragmento tiene número → None
        #    Ej: "Torres del Yacht - Juana Manso 600 - 2 Ambientes" → "Juana Manso 600"
        #    Ej: "El Faro - 3 Ambientes" → el "3" no es altura válida (< 10), descartar
        if '-' in cleaned:
            fragmentos = [f.strip() for f in cleaned.split('-')]
            candidato = None
            for frag in fragmentos:
                # Buscar fragmento con número >= 100 (alturas reales de CABA son >= 100)
                if re.search(r'\b\d{3,5}\b', frag):
                    candidato = frag
                    break
            if candidato:
                cleaned = candidato
            else:
                # Ningún fragmento tiene altura válida → nombre de edificio sin dirección
                return None, None, None

        # 3. Intentar capturar piso: "Junín 1615 piso 13"
        match_piso = re.search(
            r'([A-Za-záéíóúÁÉÍÓÚñÑü\s\.]+?)\s+(\d{2,5})\s+piso\s+(\d+)',
            cleaned, re.IGNORECASE
        )
        if match_piso:
            num = int(match_piso.group(2))
            if not (1800 <= num <= 1950 and re.search(MESES, cleaned[:match_piso.start(2)], re.IGNORECASE)):
                calle  = match_piso.group(1).strip().strip('-').strip('.').title()
                altura = match_piso.group(2).strip()
                piso   = match_piso.group(3).strip()
                return calle, altura, piso

        # 4. Iterar todos los números y encontrar la altura real
        for m in re.finditer(r'(\d{2,5})(?:[.,\s\-]|$)', cleaned):
            num = int(m.group(1))

            # Descartar números < 100: probablemente ambientes, m², baños, etc.
            if num < 100:
                continue

            # Descartar años históricos si hay un mes en el contexto previo
            contexto_previo = cleaned[:m.start()].lower()
            es_año_historico = (1800 <= num <= 1950) and bool(re.search(MESES, contexto_previo, re.IGNORECASE))
            if es_año_historico:
                continue

            # Este número es la altura real
            calle_raw = cleaned[:m.start()].strip().strip('-').strip('.').strip()
            if not calle_raw:
                continue

            calle  = calle_raw.title()
            altura = str(num)
            # piso queda None
            return calle, altura, piso

    except:
        pass

    return None, None, None

def parse_card(item):
    # Link
    href = item.get('data-to-posting', '')
    if not href:
        a = item.find('a', href=re.compile(r'/propiedades/'))
        href = a['href'] if a else ''
    link = f"https://www.zonaprop.com.ar{href}" if href.startswith('/') else href

    # Precio
    price_tag = item.find(attrs={"data-qa": "POSTING_CARD_PRICE"})
    precio_raw = clean_text(price_tag.text) if price_tag else ""
    precio, _ = parse_price(precio_raw)

    # Expensas
    exp_tag = item.find(attrs={"data-qa": "expensas"})
    expensas = clean_text(exp_tag.text) if exp_tag else None
    expensas = expensas[:-9] if expensas else None

    # ── Dirección: calle+altura y barrio van en tags separados ──
    addr_tag = item.find(class_=re.compile(r'location-address'))
    raw_address = clean_text(addr_tag.text) if addr_tag else ""
    calle, altura, piso = parse_address(raw_address)

    # Barrio/zona (el data-qa="POSTING_CARD_LOCATION")
    barrio_tag = item.find(attrs={"data-qa": "POSTING_CARD_LOCATION"})
    barrio = clean_text(barrio_tag.text) if barrio_tag else None
    barrio = barrio.split(',')[0].strip() if barrio else None

    # Características
    feat_tag = item.find(attrs={"data-qa": "POSTING_CARD_FEATURES"})
    features = clean_text(feat_tag.get_text(separator=' ')) if feat_tag else None

    # Descripción
    desc_tag = item.find(attrs={"data-qa": "POSTING_CARD_DESCRIPTION"})
    desc = clean_text(desc_tag.text) if desc_tag else None

    return link, precio, expensas, calle, altura, piso, barrio, features, desc

print("✅ Funciones utilitarias cargadas.")

✅ Funciones utilitarias cargadas.


## 3. Función principal del scrapper

In [7]:
def run_scrapper_zonaprop(enlace, operacion, max_pages=3):
    BASE = enlace
    OUTPUT_DIR = "../../data/raw"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    all_data = []
    seen_links = set()

    for page_num in range(1, max_pages + 1):
        url = f"{BASE}.html" if page_num == 1 else f"{BASE}-pagina-{page_num}.html"
        print(f"\n--- PÁGINA {page_num} ---")
        print(f"URL: {url}")

        try:
            r = cf_requests.get(
                url,
                impersonate="chrome120",
                timeout=20,
                headers={
                    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
                    "Accept-Language": "es-AR,es;q=0.9",
                    "Referer": "https://www.zonaprop.com.ar/",
                }
            )
            if r.status_code != 200:
                print(f"❌ Status {r.status_code}. Deteniendo.")
                break
        except Exception as e:
            print(f"❌ Error de red: {e}")
            break

        soup = BeautifulSoup(r.text, 'html.parser')
        items = soup.find_all('div', attrs={"data-qa": "posting PROPERTY"})

        if not items:
            print("⚠️ No se encontraron cards. Fin del listado.")
            break

        print(f"  → {len(items)} propiedades encontradas")

        for item in items:
            try:
                link, precio, expensas, calle, altura, piso, barrio, features, desc = parse_card(item)

                if not link or link in seen_links:
                    continue
                seen_links.add(link)

                print(f"  → {calle} {altura} | {precio}")
                
                all_data.append({
                    "Fecha_Scraping": date.today().isoformat(),
                    "Posting_ID":     item.get('data-id'),
                    "Sito":           'zonaprop',
                    "Operación":      operacion,
                    "Precio":         precio,
                    "Expensas":       expensas,
                    "Calle":          calle,
                    "Altura":         altura,
                    "Piso":           piso,
                    "Barrio":         barrio,       
                    "Detalles":       features,
                    "Descripción":    desc,
                    "Link":           link,
                })

            except Exception as e:
                print(f"    ⚠️ Error en card: {e}")
                continue

        time.sleep(2)

    if not all_data:
        print("⚠️ No se obtuvieron datos.")
        return None

    df = pd.DataFrame(all_data)
    features_df = df.apply(extract_smart_features, axis=1)
    df = pd.concat([df, features_df], axis=1)

    filename = f"zonaprop_{operacion}_{int(time.time())}.tsv"
    filepath = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(filepath, sep='\t', index=False, encoding='utf-8-sig')
    print(f"\n✅ ¡Éxito! Guardado en: {filepath}")
    return df

print("✅ Listo.")

✅ Listo.


## 4. ▶️ Ejecutar el scrapper

# Alquiler

In [ ]:
df_alquiler = run_scrapper_zonaprop('https://www.zonaprop.com.ar/departamentos-alquiler-capital-federal', 'alquiler', max_pages=400)


--- PÁGINA 1 ---
URL: https://www.zonaprop.com.ar/departamentos-alquiler-capital-federal.html
  → 30 propiedades encontradas
  → Mayor Arturo Luisoni 2600 | USD 450
  → Azucena Villaflor 669 | USD 6.000
  → Luis María Campos 1500 | USD 1.200
  → Colpayo 700 | USD 1.150
  → La Pampa 1100 | USD 2.200
  → None None | USD 1.300
  → Av. Libertador 2700 | USD 3.700
  → República De La India 3000 | USD 1.800
  → Paraguay 5600 | $ 600.000
  → Parana 1000 | $ 560.000
  → Macacha Guemes 330 | USD 4.600
  → Republica De Eslovenia 1900 | USD 1.000
  → Balbin 3695 | $ 690.000
  → Estomba 3700 | USD 750
  → Forest 400 | $ 650.000
  → Julián Alvarez 2360 | $ 830.000
  → Luis M Campos 1000 | $ 1.300.000
  → La Pampa 1300 | $ 890.000
  → Bernardo Irigoyen 1300 | $ 650.000
  → Comodoro Rivadavia 2320 | USD 2.000
  → Av. Del Libertador 2200 | USD 7.500
  → Salguero 3000 | USD 1.100
  → Av. Del Libertador 5700 | USD 1.100
  → Urdininea 1700 | $ 960.000
  → Paraguay 4800 | USD 1.100
  → Jose Hernandez 140

In [6]:
if df_alquiler is not None:
    print(f"Total propiedades: {len(df_alquiler)}")
    display(df_alquiler.head(10))

    cols = ["Losa_Central","Aire_Acond","Apto_Credito",
            "Cochera","Seguridad","Luminoso","Balcon_Aterrazado"]
    print("\n📊 % con cada característica:")
    print((df_alquiler[cols].mean() * 100).round(1).astype(str) + "%")

Total propiedades: 90


,Fecha_Scraping,Posting_ID,Sito,Operación,Precio,Expensas,Calle,Altura,Piso,Barrio,...,Descripción,Link,Amenities,Losa_Central,Aire_Acond,Apto_Credito,Cochera,Seguridad,Luminoso,Balcon_Aterrazado
0,2026-04-14,58601056,zonaprop,alquiler,$ 630.000,$ 150.000,Malabia,2097,None,Palermo,...,Hermoso Monoambiente en inmejorable ubicación ...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,1,0,0,0,1,0
1,2026-04-14,58708042,zonaprop,alquiler,$ 550.000,$ 140.000,Av. Belgrano,200,None,Monserrat,...,[mel-Ver datos] Departamento de dos ambientes ...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,0,0,0,0,0,0
2,2026-04-14,58420784,zonaprop,alquiler,USD 1.200,$ 375.000,Huergo,400,None,Puerto Madero,...,Broker: Lali +54 9 11 Ver datosDos ambientes e...,https://www.zonaprop.com.ar/propiedades/clasif...,4,0,1,0,1,1,1,0
3,2026-04-14,58846126,zonaprop,alquiler,$ 660.000,$ 60.000,Diaz Colodrero,3500,None,Villa Urquiza,...,Fratelli grupo inmobiliario 11 Ver datosalquil...,https://www.zonaprop.com.ar/propiedades/clasif...,3,0,0,0,0,0,1,0
4,2026-04-14,58788957,zonaprop,alquiler,USD 3.700,$ 1.068.000,Av. Del Libertador,3100,None,Palermo Chico,...,Semipiso de Categoría en Palermo Chico con Coc...,https://www.zonaprop.com.ar/propiedades/clasif...,0,1,1,0,1,1,0,0
5,2026-04-14,58798784,zonaprop,alquiler,$ 800.000,$ 80.000,Dr. Pedro Ignacio Rivera,2600,None,Belgrano,...,Monoambiente al contrafrente con balcón francé...,https://www.zonaprop.com.ar/propiedades/clasif...,3,1,0,0,0,0,0,0
6,2026-04-14,58481534,zonaprop,alquiler,$ 1.300.000,None,Conde,1839,None,Belgrano R,...,Edificio torre rodeado de jardines. 4 ambiente...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,0,0,0,0,1,0
7,2026-04-14,58787314,zonaprop,alquiler,$ 900.000,$ 184.000,Valle,1100,None,Caballito,...,Semipiso de 1 dormitorio en alquiler en Caball...,https://www.zonaprop.com.ar/propiedades/clasif...,1,0,1,0,0,0,1,0
8,2026-04-14,58791528,zonaprop,alquiler,USD 750,$ 200.000,None,None,None,Palermo,...,_ Departamento de un dormitorio en alquiler ub...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,0,0,0,0,1,0
9,2026-04-14,58770031,zonaprop,alquiler,USD 1.300,$ 407.000,None,None,None,Recoleta,...,"Luminoso departamento contrafrente, a amplio p...",https://www.zonaprop.com.ar/propiedades/clasif...,0,0,0,0,0,1,1,0



📊 % con cada característica:
Losa_Central         16.7%
Aire_Acond           42.2%
Apto_Credito          2.2%
Cochera              36.7%
Seguridad            27.8%
Luminoso             71.1%
Balcon_Aterrazado     5.6%
dtype: object


# Alquiler Temporal

In [10]:
df_alquiler_temporal = run_scrapper_zonaprop('https://www.zonaprop.com.ar/departamentos-alquiler-temporal-capital-federal', 'alquiler_temporal', max_pages=116)


--- PÁGINA 1 ---
URL: https://www.zonaprop.com.ar/departamentos-alquiler-temporal-capital-federal.html
  → 27 propiedades encontradas
  → Lola Mora 400 | USD 3.800
  → Piedras 1100 | USD 1.100
  → Av. Las Heras 2000 | USD 600
  → Rosario Vera Peñaloza 500 | USD 2.200
  → Arenales 2300 | USD 1.150
  → Av Rivadavia 5800 | USD 650
  → Av Coronel Diaz 1400 | USD 1.450
  → Nuñez 2900 | USD 650
  → Concepcion Arenal 2900 | USD 4.000
  → Marcelo T. De Alvear 500 | USD 700
  → Vera 800 | USD 700
  → Avenida Coronel Díaz 2400 | USD 1.250
  → Corrientes 1100 | USD 1.000
  → Riobamba 900 | USD 2.200
  → Emilio Ravignani 2300 | USD 1.100
  → Godoy Cruz 2900 | USD 1.400
  → Avenida Corrientes 1400 | USD 800
  → San Jose 445 | USD 550
  → Avalos 1900 | USD 750
  → Julián Álvarez 1900 | USD 700
  → Guemes 3600 | USD 800
  → Jose A Cabrera 3100 | USD 750
  → Ibera 2300 | USD 700
  → Lafinur 3200 | USD 1.900
  → Av Luis Maria Campos 300 | USD 750
  → Jorge Newberry 2500 | USD 1.200
  → Thompson 733 | 

In [11]:
if df_alquiler_temporal is not None:
    print(f"Total propiedades: {len(df_alquiler_temporal)}")
    display(df_alquiler_temporal.head(10))

    cols = ["Losa_Central","Aire_Acond","Apto_Credito",
            "Cochera","Seguridad","Luminoso","Balcon_Aterrazado"]
    print("\n📊 % con cada característica:")
    print((df_alquiler_temporal[cols].mean() * 100).round(1).astype(str) + "%")

Total propiedades: 3476


,Fecha_Scraping,Posting_ID,Sito,Operación,Precio,Expensas,Calle,Altura,Piso,Barrio,...,Descripción,Link,Amenities,Losa_Central,Aire_Acond,Apto_Credito,Cochera,Seguridad,Luminoso,Balcon_Aterrazado
0,2026-04-12,58810877,zonaprop,alquiler_temporal,USD 3.800,None,Lola Mora,400,None,Puerto Madero,...,Alquiler temporario de 3 ambientes amueblado e...,https://www.zonaprop.com.ar/propiedades/clasif...,3,0,0,0,1,1,0,0
1,2026-04-12,58731045,zonaprop,alquiler_temporal,USD 1.100,None,Piedras,1100,None,San Telmo,...,Corredor Inmobiliario responsable: Arq. Jorge ...,https://www.zonaprop.com.ar/propiedades/clasif...,5,0,0,0,0,1,1,0
2,2026-04-12,58586992,zonaprop,alquiler_temporal,USD 600,None,Av. Las Heras,2000,None,Recoleta,...,Alquiler temporario de Departamento 1 Ambiente...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,0,0,0,0,0,0
3,2026-04-12,58822120,zonaprop,alquiler_temporal,USD 2.200,None,Rosario Vera Peñaloza,500,None,Puerto Madero,...,Corredor Inmobiliario responsable: Arq. Jorge ...,https://www.zonaprop.com.ar/propiedades/clasif...,4,0,0,0,1,1,1,0
4,2026-04-12,57872253,zonaprop,alquiler_temporal,USD 1.150,None,Arenales,2300,None,Recoleta,...,Departamento de dos ambientes para alquiler te...,https://www.zonaprop.com.ar/propiedades/clasif...,5,0,1,0,1,1,1,0
5,2026-04-12,57863013,zonaprop,alquiler_temporal,USD 650,None,Av Rivadavia,5800,None,Caballito,...,Departamento de un ambiente en caballito para ...,https://www.zonaprop.com.ar/propiedades/clasif...,5,0,1,0,0,0,1,0
6,2026-04-12,58637629,zonaprop,alquiler_temporal,USD 1.450,None,Av Coronel Diaz,1400,None,Palermo,...,Departamento de tres ambientes en palermo para...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,1,0,0,0,0,0
7,2026-04-12,58584223,zonaprop,alquiler_temporal,USD 650,$ 115.000,Nuñez,2900,None,Núñez,...,Corredor Responsable: Ariel Champanier cucicba...,https://www.zonaprop.com.ar/propiedades/clasif...,3,0,1,0,0,0,1,0
8,2026-04-12,57130848,zonaprop,alquiler_temporal,USD 4.000,$ 600.000,Concepcion Arenal,2900,None,Palermo Hollywood,...,Alquiler Temporario Premium – Torre Concepción...,https://www.zonaprop.com.ar/propiedades/clasif...,4,0,0,0,1,1,1,0
9,2026-04-12,56314889,zonaprop,alquiler_temporal,USD 700,None,Marcelo T. De Alvear,500,9,Retiro,...,Departamento de 2 ambientes en alquiler tempor...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,0,0,0,0,0,0



📊 % con cada característica:
Losa_Central          7.0%
Aire_Acond           51.0%
Apto_Credito          0.2%
Cochera              16.0%
Seguridad            20.0%
Luminoso             66.4%
Balcon_Aterrazado     5.5%
dtype: object


# Venta

In [12]:
df_venta = run_scrapper_zonaprop('https://www.zonaprop.com.ar/departamentos-venta-capital-federal', 'venta', max_pages=600)


--- PÁGINA 1 ---
URL: https://www.zonaprop.com.ar/departamentos-venta-capital-federal.html
  → 25 propiedades encontradas
  → None None | USD 160.000
  → Posadas 1600 | USD 4.500.000
  → Jerónimo Salguero 700 | USD 135.000
  → Uriarte 2400 | USD 93.000
  → Malabia 2200 | USD 127.000
  → Avenida Corrientes 1900 | USD 118.130
  → Rio De Janeiro 1000 | USD 69.999
  → Tronador 2600 | USD 230.000
  → Vuelta De Obligado 3000 | USD 112.000
  → Olazabal 4100 | USD 352.000
  → Zuviria 449 | USD 226.000
  → Formosa 300 | USD 47.000
  → Rincón 1100 | USD 69.000
  → None None | USD 80.000
  → Arroyo 800 | USD 550.000
  → Monroe 4400 | USD 240.000
  → Pasteur 700 | USD 65.000
  → San Blas 2984 | USD 99.000
  → Avenida Cordoba 4200 | USD 153.000
  → Scalabrini Ortíz 2400 | USD 139.000
  → None None | USD 185.000
  → Nazca 2700 | USD 89.000
  → Av. Estado De Israel 4700 | USD 115.000
  → Contralmirante Martin Guerrico 5400 | USD 59.900
  → Av La Plata 689 | USD 846.700

--- PÁGINA 2 ---
URL: https:/

In [14]:
if df_venta is not None:
    print(f"Total propiedades: {len(df_venta)}")
    display(df_venta.head(10))

    cols = ["Losa_Central","Aire_Acond","Apto_Credito",
            "Cochera","Seguridad","Luminoso","Balcon_Aterrazado"]
    print("\n📊 % con cada característica:")
    print((df_venta[cols].mean() * 100).round(1).astype(str) + "%")

Total propiedades: 16274


,Fecha_Scraping,Posting_ID,Sito,Operación,Precio,Expensas,Calle,Altura,Piso,Barrio,...,Descripción,Link,Amenities,Losa_Central,Aire_Acond,Apto_Credito,Cochera,Seguridad,Luminoso,Balcon_Aterrazado
0,2026-04-12,58398326,zonaprop,venta,USD 160.000,$ 126.347,None,None,None,Villa Crespo,...,Hermoso tres ambientes reciclado en sexto piso...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,0,0,0,0,1,0
1,2026-04-12,57150909,zonaprop,venta,USD 4.500.000,$ 4.600.000,Posadas,1600,None,Recoleta,...,"Posadas esquina Bioy Casares, emblemático edif...",https://www.zonaprop.com.ar/propiedades/clasif...,0,0,0,0,1,1,0,0
2,2026-04-12,58790026,zonaprop,venta,USD 135.000,$ 230.000,Jerónimo Salguero,700,None,Almagro,...,"Departamento 2 ambientes con balcón al frente,...",https://www.zonaprop.com.ar/propiedades/clasif...,1,1,0,0,0,0,1,0
3,2026-04-12,58499682,zonaprop,venta,USD 93.000,$ 220.000,Uriarte,2400,None,Palermo Soho,...,Dos ambientes reciclado y amueblado (opcional)...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,0,0,0,0,1,0
4,2026-04-12,58164322,zonaprop,venta,USD 127.000,$ 230.000,Malabia,2200,None,Palermo,...,3 ambientes. Lateral. 58 M2. Vista parcial. To...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,0,0,0,0,1,0
5,2026-04-12,57689076,zonaprop,venta,USD 118.130,None,Avenida Corrientes,1900,None,Balvanera,...,Met avenue se erige en pleno centro de Buenos ...,https://www.zonaprop.com.ar/propiedades/clasif...,3,0,0,0,1,0,0,0
6,2026-04-12,58701979,zonaprop,venta,USD 69.999,$ 66.400,Rio De Janeiro,1000,None,Caballito,...,Excelente 2 ambientes en el corazón de caballi...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,0,1,0,0,1,0
7,2026-04-12,57079611,zonaprop,venta,USD 230.000,$ 325.000,Tronador,2600,None,Belgrano,...,"¡ 5 amb, balcón al frente Y cochera! ¡ en cogh...",https://www.zonaprop.com.ar/propiedades/clasif...,0,0,0,0,1,0,1,0
8,2026-04-12,55317601,zonaprop,venta,USD 112.000,None,Vuelta De Obligado,3000,None,Núñez,...,Monoambiente/ Studio de 29 m2 con salida a bal...,https://www.zonaprop.com.ar/propiedades/clasif...,3,0,1,0,0,0,1,0
9,2026-04-12,53143147,zonaprop,venta,USD 352.000,$ 170.000,Olazabal,4100,None,Villa Urquiza,...,Te presentamos este luminoso departamento de 3...,https://www.zonaprop.com.ar/propiedades/clasif...,3,0,1,0,1,0,1,0



📊 % con cada característica:
Losa_Central         14.1%
Aire_Acond           31.4%
Apto_Credito         11.8%
Cochera              42.7%
Seguridad            24.7%
Luminoso             74.4%
Balcon_Aterrazado    13.0%
dtype: object
